# TimesFM Template

Minimal starter for Google's TimesFM 2.5-200M. Load the model, forecast two dummy series, inspect the output shapes. Fork this to build your own pipeline.

In [ ]:
!pip install -q timesfm numpy matplotlib

import numpy as np
import timesfm

In [ ]:
model = timesfm.TimesFM_2p5_200M_torch.from_pretrained("google/timesfm-2.5-200m-pytorch", torch_compile=True)

model.compile(
    timesfm.ForecastConfig(
        max_context=1024,
        max_horizon=256,
        normalize_inputs=True,
        use_continuous_quantile_head=True,
        force_flip_invariance=True,
        infer_is_positive=True,
        fix_quantile_crossing=True,
    )
)

In [ ]:
inputs = [
    np.linspace(0, 1, 100),
    np.sin(np.linspace(0, 20, 67)),
]  # Two dummy inputs
point_forecast, quantile_forecast = model.forecast(horizon=12, inputs=inputs)
point_forecast.shape        # (2, 12)
quantile_forecast.shape     # (2, 12, 10): mean, then 10th to 90th quantiles.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 1, figsize=(10, 6))
labels = ['linspace(0,1,100)', 'sin(linspace(0,20,67))']
for ax, inp, label, pf in zip(axes, inputs, labels, point_forecast):
    ax.plot(range(len(inp)), inp, label='input')
    ax.plot(range(len(inp), len(inp) + len(pf)), pf, '--', label='forecast')
    ax.set_title(label)
    ax.legend()
fig.tight_layout()
fig.savefig('timesfm.png', dpi=120)
from IPython.display import Image
Image('timesfm.png')